## 0. Settings

In [2]:
# Fix root
# Asuumes the current working directory is examples/experimental/
using Pkg
Pkg.activate(normpath(joinpath(pwd(), "..", "..")))
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Projects/quantecon/ContinuousDPs.jl`


Project ContinuousDPs v0.1.0
Status `~/Projects/quantecon/ContinuousDPs.jl/Project.toml`
  [08854c51] BasisMatrices v0.8.0
  [6a86dc24] FiniteDiff v2.29.0
  [429524aa] Optim v2.0.1
  [fcd29c91] QuantEcon v0.17.1
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


In [3]:
using QuantEcon
using BasisMatrices
using ContinuousDPs
using Random
using PythonPlot
const plt = PythonPlot.pyplot;

## 1. Model

In [11]:
# Parameter Settings 
struct Params
    β::Float64
    λ::Float64
    A::Float64
    α::Float64
    δ::Float64
    ρ::Float64
    σϵ::Float64
end

p = Params(0.95, 1/3, 10.0, 0.34, 1.0, 0.90, 0.008)


Params(0.95, 0.3333333333333333, 10.0, 0.34, 1.0, 0.9, 0.008)

In [25]:
# Output
y(k, z, l, p::Params) = z * p.A * k^p.α * (1 - l)^(1 - p.α)

# Calculate consumption and k prime based on Santos equation (7.4) ("unidimensional maximization")
function c_from_l(k, z, l, p::Params)
    return z * p.A * k^p.α * (1 - l)^(-p.α) * (p.λ / (1 - p.λ)) * (1 - p.α) * l
end

function kprime_from_l(k, z, l, p::Params)
    return y(k, z, l, p) + (1 - p.δ) * k - c_from_l(k, z, l, p)
end

# Reward and transition
function f(s, l, p::Params) 
    k, logz = s
    z = exp(logz)
    if !(0 < l < 1)
        return -Inf
    end
    c = c_from_l(k, z, l, p)
    kp = kprime_from_l(k, z, l, p)
    if c <= 0 || kp < 0
        return -Inf
    end
    return p.λ*log(c) + (1 - p.λ)*log(l)
end

function g(s, l, e, p::Params)
    k, logz = s
    z = exp(logz)
    kp = kprime_from_l(k, z, l, p)
    logzp = p.ρ*logz + e
    return (kp, logzp)
end

# Domains
logz_min, logz_max = -0.32, 0.32
k_min, k_max = 0.10, 10.0
x_lb(s), x_ub(s) = 1e-10, 1 - 1e-10

(1.0e-10, 0.9999999999)

In [ ]:
# shocks, weights (Gauss-Hermite quadrature)
n_shocks = 7
shocks, weights = qnwnorm(n_shocks, 0.0, p.σϵ^2)

([-0.03000351774180594, -0.01893407528587633, -0.009235243157919746, 0.0, 0.009235243157919746, 0.01893407528587633, 0.03000351774180594], [0.0005482688559722181, 0.030757123967586456, 0.24012317860501223, 0.4571428571428571, 0.24012317860501223, 0.030757123967586456, 0.0005482688559722181])

In [26]:
# interp (Chebyshev polynomial)
nk, nlogz = 143, 9 # based on Santos (1999) Table 16
basis = Basis(
    ChebParams(nk, k_min, k_max),
    ChebParams(nlogz, logz_min, logz_max)
)

2 dimensional Basis on the hypercube formed by (0.1, -0.32) × (10.0, 0.32).
Basis families are Cheb × Cheb
